# 03 · Browser prototype

Generate a standalone browser session from the task bank and an assigned sequence. The session exports JSON for Notebook 04.

Run the setup cell first. See the repository README for ZIP and GitHub setup.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile
from zipfile import ZipFile

REPO_URL = ""
PROJECT_DIR = Path.cwd()

def find_project(directory):
    candidates = [directory, *sorted(path for path in directory.iterdir() if path.is_dir())]
    return next((path for path in candidates if (path / "src" / "pilot.py").is_file()), None)

project = find_project(PROJECT_DIR)
if project is None:
    if REPO_URL:
        project = Path(tempfile.mkdtemp(prefix="python-hci-")) / "repo"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project)], check=True)
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Extract the repository and set PROJECT_DIR to its root.") from exc
        import io
        uploaded = files.upload()
        archives = [data for name, data in uploaded.items() if name.lower().endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload one project ZIP.")
        directory = Path(tempfile.mkdtemp(prefix="python-hci-"))
        with ZipFile(io.BytesIO(archives[0])) as archive:
            archive.extractall(directory)
        project = find_project(directory)
if project is None or not (project / "study" / "stimuli.json").is_file():
    raise FileNotFoundError("Project root not found. Use the ZIP supplied with these notebooks.")
sys.path.insert(0, str(project.resolve()))
from src import pilot
OUT = project / "outputs"
print("Project root:", project)

In [ ]:
MODE = "demo"
PARTICIPANT_ID = "WALKTHROUGH"
SEQUENCE = 1

session = pilot.build_session(OUT / f"{MODE}-session.html", PARTICIPANT_ID, SEQUENCE, MODE)
print("Saved:", session)
print("Trial order:", pilot.schedule(SEQUENCE))

Open the generated HTML in a separate browser tab. In Colab, use the left **Files** pane to download it, or set `DOWNLOAD = True` below. Work through all three rounds and download the response JSON after the debrief.

A participant session requires the applicable institutional arrangements and an assigned study ID. Use the sequence from Notebook 01, change `MODE` to `"participant"`, and store response files in access-controlled storage. The HTML runs locally and does not transmit responses. Refreshing clears the in-progress session.

In [ ]:
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files
    files.download(str(session))